In [ ]:
%pylab inline
import eucare as ec

In [ ]:

def hyperbolic_band_graph(G=None, dists=(0.05, 0.05), band_length=1, dual=False):
    import eucare as ec
    
    if not isinstance(dists, (list, tuple)):
        dists = [dists, dists]
        
    def wrapped_elliptic_f(z, m):
        return complex(N(elliptic_f(z, m)))

    def complex_to_array(z):
        return np.array([z.real, z.imag])

    def disk_to_square(z, w=1):
        return np.sqrt(1j) * wrapped_elliptic_f(np.arcsin(w * z), -1)
        #return np.sqrt(2) * wrapped_elliptic_f(np.arcsin(np.sqrt(z+1)), np.sqrt(2)/2)

    def disk_to_halfplane(z):
        return (z + 1j) / (1j * z + 1)
    
    def to_band(z):
        return complex_to_array(np.arctanh(z) * 2 / np.pi)

    if G is None:
        G = ec.example_graphs.from_tiles(ec.example_graphs.curved_platonic(7, 3), 1)

    def map_to_square(G):
        G_square = G.copy()
        G_square.geometry = ec.geometries.EuclideanGeometry
        for v in G_square.vertices:
            if 'square_pos' in v:
                v['pos'] = v['square_pos']
            else:
                v['pos'] = to_band(v['pos'])
        return G_square


    nv_before = None
    nv_after = G.order
    while nv_before != nv_after:
        nv_before = nv_after
        print(nv_before)
        for v in G.vertices:
            if 'square_pos' not in v:
                v['square_pos'] = to_band(v['pos'])
                for e in v.outgoing_iter():
                    if 'square_pos' not in e.dest:
                        continue
                    e['square_length'] = e.rev['square_length'] = np.linalg.norm(v['square_pos'] - e.dest['square_pos'])

        queue = {}
        for v in G.vertices:
            if not v.on_border():
                continue
            if np.max(np.abs(v['square_pos'][0])) > band_length / 2:
                continue
            v['square_length'] = np.mean([e['square_length'] for e in v.outgoing_iter() if e.on_border()])
            
            queue[v] = 0.5-np.abs(v['square_pos'][1])
            queue[v] = queue[v] - (dists[0 if v['square_pos'][1] > 0 else 1])
            
        for v, dist in reversed(sorted(queue.items(), key=lambda kv: kv[1])):
            if dist > 0:
                if v.on_border():
                    ec.example_graphs.complete_vertex(G, v)
        nv_after = G.order

    if dual:
        G = ec.conway.dual_graph()(G)
    G_square = map_to_square(G)

    return G_square

G = hyperbolic_band_graph(dists=(0.1, 0.3), band_length=2)
G.show()

In [ ]:
def make_G0():
    G = ec.example_graphs.from_tiles(ec.example_graphs.curved_platonic(7, 3), 0)
    h = next(h for h in G.halfedges 
             if h.on_border() and h.orig['pos'].real < 0 and h.orig['pos'].imag < 0 and h.dest['pos'].imag > 0)
    # G.execute_edge_instruction(h)

    geom = G.geometry

    m1 = geom.center_of_mass(np.array([h.orig['pos'], h.dest['pos']]))
    m2 = geom.center_of_mass(np.array([h.nex.orig['pos'], h.nex.dest['pos']]))
    m = geom.center_of_mass(np.array([m1, m2]))
    translate = geom.translation(m, geom.origin())
    # translate = geom.translation(h.pre.pre.pre.orig['pos'], geom.origin())

    rotate = geom.rotation(0, 0)
    for v in G.vertices:
        v['pos'] = rotate(translate(v['pos']))


    m1 = geom.center_of_mass(np.array([h.orig['pos'], h.dest['pos']]))
    m2 = geom.center_of_mass(np.array([h.nex.orig['pos'], h.nex.dest['pos']]))
    rotate = geom.rotation(0, -geom.angle_to_axis(m1))
    for v in G.vertices:
        v['pos'] = rotate(v['pos'])


    return G

G = hyperbolic_band_graph(make_G0(), dists=(2, 2))
G.show()
ps, _ = G.get_position_view()
period = np.max(ps[:, 0]) - np.min(ps[:, 0])
print(period)
# G = hyperbolic_band_graph(G, dists=(0.1, 0.3), band_length=3)
# G.show()

In [ ]:
import networkx as nx
from eucare.conversions import EHEG_from_nx
from eucare.overlap import group_closeby, remove_duplicates


In [ ]:
period

In [ ]:


# period = 0.9225096200947644
# n = 7
# G = hyperbolic_band_graph(dists=(0.09, 0.06), band_length=period * n + 1, dual=True)


# period = 0.9225096200947644
# n=7
# G = hyperbolic_band_graph(dists=(0.06, 0.027), band_length=period * n + 1, dual=True)

n = 18
G = hyperbolic_band_graph(make_G0(), dists=(0.07, 0.025), band_length=period + 0.4, dual=False)
G.show()

ps, vs = G.get_position_view()

k = ps.copy()
k = np.array([complex(*ki) for ki in k])

k *= 1j

zero_points = sorted((k.imag[np.abs(k.real) < 1e-6]))
# period = zero_points[2] - zero_points[0]
# print(period)

k -= np.mean(k)
k -= np.min(k.real)
k = np.exp(k / (period * n) * 2 * np.pi)


Gtotal = ec.half.GeometricHEG()
for i in range(n):
    ps, _ = G.get_position_view()
    ps[:] = np.stack([k.real, k.imag], axis=-1)
    Gtotal.add_graph(G.copy())
    k *= np.exp(2j*np.pi/n)
G = Gtotal

# normalize positions
ps, vs = G.get_position_view()
k = ps.copy()
k = np.array([complex(*ki) for ki in k])
k *= 1 / np.std(np.abs(k))
k = np.stack([k.real, k.imag], axis=-1)
ps[:] = k

G = remove_duplicates(G, eps=1e-6)
G.delete_subset([f for f in G.faces if f.order() > 7])
# G.delete_subset(central_face(G))
# G.delete_subset([v for v in G.vertices if v.on_border() and v.order() == 2])
# G.delete_subset([f for f in G.faces if f.order() > 7])
G.show(line_width=0.01, height=1000)

In [ ]:
ec.io.save_graph('graphs/hyperbolic_annulus_7_smooth', G, overwrite=True)

In [ ]:
ec.io.load_graph('graphs/hyperbolic_annulus_7.heg').show()

In [ ]:


# period = 0.9225096200947644
# n = 7
# G = hyperbolic_band_graph(dists=(0.09, 0.06), band_length=period * n + 1, dual=True)


# period = 0.9225096200947644
# n=7
# G = hyperbolic_band_graph(dists=(0.06, 0.027), band_length=period * n + 1, dual=True)

n = 18
period = 4.732050807568882
delete_central = True

G = ec.example_graphs.from_tiles(ec.example_tilesets.curved_omnitruncate(6, 3), rings=3)
ps, vs = G.get_position_view()
ps[:] = ps @ ec.base.rotation_matrix(np.pi/12)

# ps = ps[ps[:, 1] > 0]
# ps = ps[ps[:, 1] < 1]
# ps = np.sort(ps[:, 0])
# period = ps[2] - ps[0]
# print(period)

G = ec.conway.dual_graph()(G)
G.show()

ps, vs = G.get_position_view()

k = ps.copy()
k = np.array([complex(*ki) for ki in k])

k *= 1j

zero_points = sorted((k.imag[np.abs(k.real) < 1e-6]))
# period = zero_points[2] - zero_points[0]
# print(period)

k -= np.mean(k)
k -= np.min(k.real)
k = np.exp(k / (period * n) * 2 * np.pi)


Gtotal = ec.half.GeometricHEG()
for i in range(n):
    ps, _ = G.get_position_view()
    ps[:] = np.stack([k.real, k.imag], axis=-1)
    Gtotal.add_graph(G.copy())
    k *= np.exp(2j*np.pi/n)
G = Gtotal

# normalize positions
ps, vs = G.get_position_view()
k = ps.copy()
k = np.array([complex(*ki) for ki in k])
# k *= 1 / np.std(np.abs(k))
k = np.stack([k.real, k.imag], axis=-1)
k /= np.std(k)
ps[:] = k


G = remove_duplicates(G, eps=1e-6)


# for v in list(G.vertices):
#     if v in G.vertices and v.order() == 2:
#         G.join_vertex(v)
        
if delete_central:
    G.delete_subset([f for f in G.faces if f.order() > 12])
else:
    G.delete_subset([f for f in G.faces if f.area() < 0])
    
# G.delete_subset(central_face(G))
# G.delete_subset([v for v in G.vertices if v.on_border() and v.order() == 2])
# G.delete_subset([f for f in G.faces if f.order() > 7])
G.show(line_width=0.01, height=1000)

In [ ]:
ec.io.save_graph('graphs/ring_12.6.4_00', G, overwrite=True)